# Chapter 2. Language as a sign system; text as numbers
### 2-bob. Til belgilar tizimi sifatida va mashina uchun til

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dilrabonu/ai-linguistics-uz/blob/main/notebooks/02_text_as_numbers.ipynb)

> **Status: draft.**

**You will learn to:** (1) see why one Uzbek word can be four different byte sequences, (2) normalize Uzbek apostrophes, (3) check Zipf's law on Uzbek text, (4) compute TF-IDF.

**Siz o'rganasiz:** bitta o'zbekcha so'z nima uchun mashina uchun to'rt xil bo'lishi mumkinligini, apostrofni normallashtirishni, Sipf qonunini va TF-IDF'ni.

In [ ]:
# Setup: installs the course utilities (takes ~20 s on Colab)
%pip install -q "uzling @ git+https://github.com/dilrabonu/ai-linguistics-uz"

## 1. One word, four byte sequences

Inson uchun bitta so'z, mashina uchun to'rt xil ketma-ketlik.

In [ ]:
import unicodedata

spellings = ["o'zbek", "o\u02bbzbek", "o\u2018zbek", "o`zbek"]
for s in spellings:
    mark = s[1]
    print(f"{s:10} U+{ord(mark):04X}  {unicodedata.name(mark):35} bytes={s.encode('utf-8')}")

print("\nDistinct strings:", len(set(spellings)))

## 2. Normalization with `uzling`

`normalize_uz` o'/g' uchun U+02BB, tutuq belgisi (ma'no) uchun U+02BC belgisini qo'yadi.

In [ ]:
from uzling import apostrophe_profile, normalize_uz

print("After normalization:", len({normalize_uz(s) for s in spellings}))

sample = "O`zbekiston ma'no-mazmuni, o\u2018qituvchi va g'isht. Sun`iy intellekt ta\u2019limda."
print(apostrophe_profile(sample))
print(normalize_uz(sample))

## 3. Zipf's law on Uzbek Wikipedia

We stream 2,000 articles (nothing is stored) and compare vocabulary size before and after normalization.

In [ ]:
import re
from collections import Counter
from itertools import islice

from datasets import load_dataset

ds = load_dataset("wikimedia/wikipedia", "20231101.uz", split="train", streaming=True)
texts = [row["text"] for row in islice(ds, 2000)]

WORD = re.compile(r"[^\W\d_]+(?:['`\u2018\u2019\u02bb\u02bc][^\W\d_]+)*")

def vocab(docs):
    c = Counter()
    for d in docs:
        c.update(w.lower() for w in WORD.findall(d))
    return c

raw = vocab(texts)
norm = vocab(normalize_uz(t) for t in texts)
print(f"Vocabulary before: {len(raw):,}  after: {len(norm):,}  (-{1 - len(norm)/len(raw):.1%})")
print(norm.most_common(15))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

freqs = np.array(sorted(norm.values(), reverse=True))
ranks = np.arange(1, len(freqs) + 1)
plt.figure(figsize=(6, 4))
plt.loglog(ranks, freqs, label="Uzbek Wikipedia (2,000 articles)")
plt.loglog(ranks, freqs[0] / ranks, "--", label="ideal Zipf: f ~ 1/r")
plt.xlabel("rank")
plt.ylabel("frequency")
plt.title("Zipf's law in Uzbek")
plt.legend()
plt.tight_layout()
plt.show()

## 4. TF-IDF on three short Uzbek texts

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

docs = [
    "Terma jamoa futbol bo'yicha jahon chempionatiga yo'llanma oldi.",
    "Markaziy bank asosiy stavkani o'zgarishsiz qoldirdi, inflyatsiya pasaymoqda.",
    "Universitet talabalari uchun sun'iy intellekt bo'yicha yangi kurs ochildi.",
]
docs = [normalize_uz(d) for d in docs]
vec = TfidfVectorizer(token_pattern=WORD.pattern, lowercase=True)
X = vec.fit_transform(docs)
terms = vec.get_feature_names_out()
for i, row in enumerate(X.toarray()):
    top = row.argsort()[::-1][:4]
    print(f"doc {i+1}:", [terms[j] for j in top])

## Exercises / Topshiriqlar

1. Collect 50 Uzbek comments from social media and run `apostrophe_profile`. Which character is most common?
2. Why is it risky to normalize the tutuq belgisi (ma'no) and the o'/g' apostrophe to the same character?
3. How does the Uzbek Zipf curve differ from English? Relate your answer to agglutination.
4. Add a fourth document to the TF-IDF example and explain how the top terms change.